In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from jax_sn.operators.transport_operator import TransportOperatorBatched
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source

In [43]:
def setup_base_case(dim, degree):
    n_elem = 5
    n_other_dim = 10
    other_dims = [[(2.0, n_other_dim)]]*(dim - 1)
    xs0 = CrossSectionData.constant(0.2, 1, 1)
    verts, conn, cross_sections, sources = create_cartesian_setup(2, [[(1.0, n_elem)]] + other_dims, [xs0], [1.0], 0)
    element_type                         = BasixLagrangianSimplex(degree, dim)
    domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections)    
    solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)

    # convert source to Source class
    sources = jnp.repeat(sources, repeats = solution_domain.n_basis, axis = -1).reshape(-1, solution_domain.n_basis)

    return solution_domain, sources


solution_domain, sources = setup_base_case(2,3)

i_tn = 50
angle_i = jsn.quadrature_set.create_tn_quadrature_set(4)[50]
toposort = TopologicalSortedAngle.from_domain_angle(solution_domain.domain, angle_i[0], 100)

In [44]:
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.operators.mass_operator.mass_operator_base import _mass_kernel_idx_in_precomp_vmap
from jax_sn.operators.transport_operator import TransportOperatorBatched, transport_inverse_sweep_batched

In [45]:
transport_operator = TransportOperatorBatched.from_solution_domain_data_and_toposort(solution_domain.solution_domain_data, toposort, batch_size = 10)
mass_operator = MassOperator.from_fem_matrices_and_precomputed_geometric_data(solution_domain.fem_matrices, solution_domain.precomputed_geometric_data)

In [ ]:
jnp.

In [95]:
def test_transport_operator_adjoint(transport_operator):    
    onp.random.seed(12122025)


    n_elements = transport_operator.solution_domain_data.domain.n_elements        
    n_basis    = transport_operator.solution_domain_data.n_basis
    if transport_operator.batched_angles:
        n_angles = transport_operator.topological_sorted_angles.angle.shape[0]
        x = jnp.array(onp.random.random(size = (n_angles, n_elements,  n_basis)))
        y = jnp.array(onp.random.random(size = (n_angles, n_elements,  n_basis)))
    else:
        x = jnp.array(onp.random.random(size = (n_elements,  n_basis,)))
        y = jnp.array(onp.random.random(size = (n_elements,  n_basis,)))

    Tx = transport_operator(x, 0)
    Tty = transport_operator.T(y, 0)
    lhs = jnp.dot(Tx.ravel(), y.ravel())
    rhs = jnp.dot(x.ravel(), Tty.ravel())
    onp.testing.assert_allclose(lhs, rhs, rtol = 1e-12)
    
    Tinvx = transport_operator.inv(x, 0)
    Ttinv_y = transport_operator.T_inv(y, 0)
    lhs_inv = jnp.dot(Tinvx.ravel(), y.ravel())
    rhs_inv = jnp.dot(x.ravel(), Ttinv_y.ravel())
    onp.testing.assert_allclose(lhs_inv, rhs_inv, rtol = 1e-12)
    
test_transport_operator_adjoint(transport_operator)

In [80]:
random_number = onp.random.random(size = sources.shape)
source_x = mass_operator(sources)

In [81]:
aaaaaa = transport_operator.T(random_number,0).ravel()

In [82]:
jnp.dot(source_x.ravel(), aaaaaa)

Array(0.00232205, dtype=float64)

In [83]:
bbbb = transport_operator(source_x, 0 ).ravel()


jnp.dot(bbbb, random_number.ravel())

Array(0.00232205, dtype=float64)

In [86]:
transport_operator.T_inv(source_x, 0)

Array([[6.62423919e-08, 6.62423820e-08, 3.34682622e-01, 9.48345059e-02, 2.44485150e-01, 9.48345059e-02, 2.44485150e-01, 6.62423940e-08, 6.62423950e-08, 1.14146927e-01],
       [3.28928601e-07, 3.34682556e-01, 3.34682555e-01, 3.34682555e-01, 3.34682556e-01, 9.48344757e-02, 2.44485179e-01, 9.48344765e-02, 2.44485179e-01, 2.25687967e-01],
       [6.62423919e-08, 6.62423820e-08, 3.34682622e-01, 9.48345059e-02, 2.44485150e-01, 9.48345059e-02, 2.44485150e-01, 6.62423940e-08, 6.62423950e-08, 1.14146927e-01],
       [3.28928602e-07, 3.34682556e-01, 3.34682555e-01, 3.34682555e-01, 3.34682556e-01, 9.48344757e-02, 2.44485179e-01, 9.48344765e-02, 2.44485179e-01, 2.25687967e-01],
       [6.62423921e-08, 6.62423822e-08, 3.34682622e-01, 9.48345059e-02, 2.44485150e-01, 9.48345059e-02, 2.44485150e-01, 6.62423940e-08, 6.62423951e-08, 1.14146927e-01],
       [3.28928601e-07, 3.34682556e-01, 3.34682555e-01, 3.34682555e-01, 3.34682556e-01, 9.48344757e-02, 2.44485179e-01, 9.48344765e-02, 2.44485179e-01, 2.2

In [41]:
import equinox as eqx
@eqx.filter_jit
def okay(transport_operator, mass_operator, sources):
    return transport_operator.inv(mass_operator(sources), 0)

In [42]:
%timeit okay(transport_operator, mass_operator, sources).block_until_ready()#transport_operator.inv(mass_operator(sources), 0)

4.28 ms ± 237 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
%timeit transport_operator.inv(mass_operator(sources), 0).block_until_ready()

12.9 ms ± 313 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
